# SegFormer-B2 Maritime Segmentation (Colab)

Train on **LaRS + MaSTr1325** from Google Drive.  
Data: `MaritimeSegmentation/datasets/` (lars, mastr1325).  
Checkpoints: `MaritimeSegmentation/checkpoints/`.

Set runtime to **GPU** (T4). Run cells in order.

In [ ]:
# Mount Drive and set paths
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

# Base folder on Drive: MaritimeSegmentation
DRIVE_BASE = Path("/content/drive/MyDrive/MaritimeSegmentation")
DATASETS_ROOT = DRIVE_BASE / "datasets"
CHECKPOINTS_DIR = DRIVE_BASE / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

LARS_IMAGES = DATASETS_ROOT / "lars" / "lars_images"
LARS_ANNOTATIONS = DATASETS_ROOT / "lars" / "lars_annotations"
MASTR_IMAGES = DATASETS_ROOT / "mastr1325" / "MaSTr1325_images_512x384"
MASTR_MASKS = DATASETS_ROOT / "mastr1325" / "MaSTr1325_masks_512x384"

print("LARS exists:", LARS_IMAGES.exists())
print("MaSTr exists:", MASTR_IMAGES.exists())
print("Checkpoints dir:", CHECKPOINTS_DIR)

In [ ]:
!pip install -q transformers timm albumentations

In [ ]:
# Config (same as project)
NUM_CLASSES = 4
CLASS_NAMES = ("Sky", "Water", "Land", "Obstacle")
IGNORE_INDEX = 255
LARS_TO_UNIFIED = {0: 3, 1: 1, 2: 0}
MASTR_TO_UNIFIED = {0: 2, 1: 1, 2: 0}
MASTR_IGNORE_VALUE = 4
BATCH_SIZE = 8
EPOCHS = 60
LR = 6e-5
WEIGHT_DECAY = 0.01
INPUT_HEIGHT, INPUT_WIDTH = 384, 512
NUM_WORKERS = 2
SAVE_EVERY_N_EPOCHS = 5
VAL_EVERY_N_EPOCHS = 1

In [ ]:
# Dataset classes
import json
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, ConcatDataset

def _map_mastr_mask(mask):
    out = np.full_like(mask, IGNORE_INDEX, dtype=np.int64)
    for src, dst in MASTR_TO_UNIFIED.items():
        out[mask == src] = dst
    out[mask == MASTR_IGNORE_VALUE] = IGNORE_INDEX
    return out

def _map_lars_mask(mask):
    out = np.full_like(mask, IGNORE_INDEX, dtype=np.int64)
    for src, dst in LARS_TO_UNIFIED.items():
        out[mask == src] = dst
    out[mask == 255] = IGNORE_INDEX
    return out

class MaSTr1325Dataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.transform = transform
        self.samples = []
        for p in sorted(self.images_dir.glob("*.jpg")):
            mask_path = self.masks_dir / f"{p.stem}m.png"
            if mask_path.exists():
                self.samples.append((p, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        mask = _map_mastr_mask(np.array(Image.open(mask_path)))
        if self.transform:
            out = self.transform(image=image, mask=mask)
            image, mask = out["image"], out["mask"]
        mask = torch.from_numpy(mask).long() if isinstance(mask, np.ndarray) else mask.long()
        return {"image": image, "mask": mask}

class LaRSDataset(Dataset):
    def __init__(self, split, images_root, annotations_root, transform=None):
        self.images_dir = Path(images_root) / split / "images"
        self.masks_dir = Path(annotations_root) / split / "semantic_masks"
        self.transform = transform
        with open(Path(annotations_root) / split / "image_annotations.json") as f:
            data = json.load(f)
        self.samples = []
        for a in data.get("annotations", []):
            fn = a["file_name"]
            img_path = self.images_dir / fn
            mask_path = self.masks_dir / (Path(fn).stem + ".png")
            if img_path.exists() and mask_path.exists():
                self.samples.append((img_path, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = np.array(Image.open(img_path).convert("RGB"))
        mask = _map_lars_mask(np.array(Image.open(mask_path)))
        if self.transform:
            out = self.transform(image=image, mask=mask)
            image, mask = out["image"], out["mask"]
        mask = torch.from_numpy(mask).long() if isinstance(mask, np.ndarray) else mask.long()
        return {"image": image, "mask": mask}

def CombinedMaritimeDataset(split, transform=None, use_mastr=True, use_lars=True):
    datasets = []
    if use_mastr and MASTR_IMAGES.exists():
        full = MaSTr1325Dataset(MASTR_IMAGES, MASTR_MASKS, transform)
        n = len(full)
        if n > 0:
            val_size = max(1, n // 10)
            idx = range(0, n - val_size) if split == "train" else range(n - val_size, n)
            datasets.append(torch.utils.data.Subset(full, idx))
    if use_lars and LARS_IMAGES.exists():
        ds = LaRSDataset(split, LARS_IMAGES, LARS_ANNOTATIONS, transform)
        if len(ds) > 0:
            datasets.append(ds)
    if not datasets:
        raise FileNotFoundError("No data. Check Drive path: MaritimeSegmentation/datasets/lars and mastr1325")
    return ConcatDataset(datasets)

In [ ]:
# Transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tf = A.Compose([
    A.Resize(INPUT_HEIGHT, INPUT_WIDTH),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.OneOf([A.MotionBlur(p=0.3), A.GaussianBlur(blur_limit=3, p=0.3)], p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(transpose_mask=True),
])
val_tf = A.Compose([
    A.Resize(INPUT_HEIGHT, INPUT_WIDTH),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(transpose_mask=True),
])

train_ds = CombinedMaritimeDataset("train", transform=train_tf)
val_ds = CombinedMaritimeDataset("val", transform=val_tf)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
# Model
from transformers import SegformerForSemanticSegmentation

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Device: {device}")

In [ ]:
# Training loop
import torch.nn as nn
import torch.nn.functional as F

criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda e: (1 - e / EPOCHS) ** 0.9)

best_miou = 0.0
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)
        optimizer.zero_grad()
        out = model(pixel_values=images)
        logits = F.interpolate(out.logits, size=masks.shape[1:], mode="bilinear", align_corners=False)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()
    train_loss /= len(train_loader)

    if (epoch + 1) % VAL_EVERY_N_EPOCHS == 0:
        model.eval()
        val_loss = 0.0
        correct = total_pixels = 0
        class_correct = [0] * NUM_CLASSES
        class_total = [0] * NUM_CLASSES
        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(device)
                masks = batch["mask"].to(device)
                out = model(pixel_values=images)
                logits = F.interpolate(out.logits, size=masks.shape[1:], mode="bilinear", align_corners=False)
                val_loss += criterion(logits, masks).item()
                pred = logits.argmax(dim=1)
                valid = masks != IGNORE_INDEX
                correct += (pred[valid] == masks[valid]).sum().item()
                total_pixels += valid.sum().item()
                for c in range(NUM_CLASSES):
                    m = masks == c
                    if m.any():
                        class_total[c] += m.sum().item()
                        class_correct[c] += (pred[m] == c).sum().item()
        val_loss /= max(len(val_loader), 1)
        acc = correct / max(total_pixels, 1)
        ious = [class_correct[c] / class_total[c] for c in range(NUM_CLASSES) if class_total[c] > 0]
        miou = sum(ious) / max(len(ious), 1) if ious else 0.0
        print(f"Epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  acc={acc:.4f}  mIoU={miou:.4f}")

        if miou > best_miou:
            best_miou = miou
            ckpt_path = CHECKPOINTS_DIR / "best_segformer_b2_maritime.pt"
            torch.save({"epoch": epoch+1, "model_state_dict": model.state_dict(), "miou": miou}, ckpt_path)
            print(f"  -> saved best to {ckpt_path}")

    if (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0:
        torch.save({"epoch": epoch+1, "model_state_dict": model.state_dict()}, CHECKPOINTS_DIR / f"segformer_b2_epoch_{epoch+1}.pt")

print(f"Done. Best mIoU: {best_miou:.4f}")